In [ ]:
import pandas as aju
import numpy as np
import seaborn as sns
import warnings
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier,plot_tree
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split,cross_val_score

In [ ]:
warnings.filterwarnings('ignore')

In [ ]:
df = aju.read_csv('../DATASETS/ML_08_DecisionTree_CleavelandHeart.csv',header = None)

In [ ]:
df.head()

In [ ]:
df.columns = ['age','sex','cp','restbp','chol','fbs','restecg','thalach','exang',
              'oldpeak','slope','ca','thal','hd']

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df.describe()

In [ ]:
df.age.unique()

In [ ]:
df.sex.unique()

In [ ]:
df.restbp.unique()

In [ ]:
df.chol.unique()

In [ ]:
df.fbs.unique()

In [ ]:
df.restecg.unique()

In [ ]:
df.thalach.unique()

In [ ]:
df.hd.unique()

In [ ]:
df.slope.unique()

In [ ]:
df.oldpeak.unique()

In [ ]:
df.exang.unique()

In [ ]:
df.ca.unique()

In [ ]:
df.thal.unique()

In [ ]:
len(df[df.thal == '?'])

In [ ]:
len(df[df.ca == '?'])

In [ ]:
df1 = df[(df.ca != '?')&(df.thal != '?')]

In [ ]:
df1.shape

In [ ]:
x = df1.drop('hd',axis = 'columns')
y = df1.hd

In [ ]:
x

In [ ]:
y

In [ ]:
x.dtypes

In [ ]:
x.cp.unique()

In [ ]:
{1,2} <= {1,2,3,4,5}

In [ ]:
df2 = aju.get_dummies(x, columns=['cp', 'restecg', 'slope', 'thal'])

df3 = df2.apply(lambda col: col.astype(int) if set(col.unique()) <= {0, 1, 0.0, 1.0, '0.0', '1.0'} else col)


In [ ]:
df3.head()

In [ ]:
df3.columns

In [ ]:
y.unique()

In [ ]:
y_not_zero = y>0
y[y_not_zero] = 1
y.unique()

In [ ]:
x_train,x_test,y_train,y_test = train_test_split(df3,y,random_state = 99)

In [ ]:
print(x_train.shape,y_train.shape,x_test.shape,y_test.shape)

In [ ]:
clf = DecisionTreeClassifier(random_state = 99)
clf_dt = clf.fit(x_train,y_train)

In [ ]:
plt.figure(figsize = (15,8))
plot_tree(clf_dt,filled = True,rounded = True,class_names = ['No HD','Yes HD'],
         feature_names = df3.columns)
plt.show()

In [ ]:
y_pred = clf_dt.predict(x_test)

In [ ]:
confusion_matrix(y_test,y_pred)

In [ ]:
path  = clf_dt.cost_complexity_pruning_path(x_train,y_train)
ccp_alpha = path.ccp_alphas

In [ ]:
path

In [ ]:
ccp_alpha

In [ ]:
ccp_alphas = ccp_alpha[:-1]
ccp_alphas

In [ ]:
clf_dts = []
for i in ccp_alphas:
    clf_dt = DecisionTreeClassifier(random_state = 99 ,ccp_alpha = i)
    clf_dt.fit(x_train,y_train)
    clf_dts.append(clf_dt)

In [ ]:
train_scores = [clf_dt.score(x_train,y_train) for clf_dt in clf_dts]

In [ ]:
train_scores

In [ ]:
test_scores = [clf_dt.score(x_test,y_test) for clf_dt in clf_dts]
test_scores

In [ ]:
fig, ax = plt.subplots()
ax.set_xlabel('alpha')
ax.set_ylabel('accuracy')
ax.set_title('Acc vs Alpha')
ax.plot(ccp_alphas,train_scores,marker = 'o',label = 'train',drawstyle = 'steps-post')
ax.plot(ccp_alphas,test_scores,marker = 'o',label = 'test',drawstyle = 'steps-post')
ax.legend()
plt.show()

In [ ]:
clf_dt = DecisionTreeClassifier(random_state = 99,ccp_alpha = 0.008)
scores = cross_val_score(clf_dt,x_train,y_train,cv = 5)
scores

In [ ]:
df5 = aju.DataFrame(data = {'tree':range (5),'accuracy':scores})
df5.plot(x='tree',y='accuracy',marker = 'o',linestyle='--')

In [ ]:
alpha_loop_values = []
for i in ccp_alphas:
    clf_dt = DecisionTreeClassifier(random_state = 99, ccp_alpha = i)
    scores = cross_val_score(clf_dt,x_train,y_train,cv = 5)
    alpha_loop_values.append([i,np.mean(scores),np.std(scores)])

alpha_res = aju.DataFrame(alpha_loop_values,columns=['alpha','mean_acc','std'])
alpha_res

In [ ]:
ideal_ccp = alpha_res[alpha_res.mean_acc == alpha_res.mean_acc.max()]['alpha']

In [ ]:
# alpha_res[(alpha_res['alpha']>0.012)&(alpha_res['alpha']<0.015)]

In [ ]:
ideal_cpp = float(ideal_ccp)
ideal_cpp

In [ ]:
clf_prune = DecisionTreeClassifier(random_state = 99,ccp_alpha = ideal_cpp)
clf_prune.fit(x_train,y_train)

In [ ]:
ConfusionMatrixDisplay.from_estimator(clf_prune,x_test,y_test)

In [ ]:
plot_tree(clf_prune,filled= True,rounded = True,class_names=['no','yes'],feature_names = x_train.columns)
plt.show()